# Canário 4128 | ME3N

| Item | Valor |
|---|---|
| Objeto | ``dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`` (tabela) |
| Ambiente | DEV, leitura |
| Schema | `DESCRIBE_ME3N.csv`, 48 colunas; clustering: num_documento_compras, cod_material, cod_centro, dt_fim_validade |
| Filtro Datalake | `cod_centro = '4128'` |
| Filtro SAP | Centro 4128; confirmar outros filtros e mesmo período |
| Volume esperado da exploratória | 908 linhas |
| Chave observada no universo | num_documento_compras + num_item_documento_compras; 707707 linhas e 0 grupos repetidos |
| Campos 100% vazios/nulos no universo sem lacuna explícita | 0 (investigar no SAP, não afirmar defeito) |

**Alertas:** dados pessoais mascarados; valores DOUBLE exigem tolerância; snapshot pode mudar. **Como usar:** importar e executar Run All. SQL independente e somente leitura. Nunca usar SELECT * para dados. Fonte dos resultados: HTML executado de `ME3N`.

### Índice
1 Metadados; 2 Volume; 3 Centro; 4 Chave; 5 Duplicidade; 6 Preenchimento; 7 Cardinalidade; 8 Domínio; 9 Indicadores; 10 Perfil numérico; 11 Totais; 12 Datas; 13 Códigos; 14 Checksum; 15 Amostra; 16 Distribuição; 17 Freshness; 18 Regras; 19 Amostra ampliada; 20 Resumo; 21 Extração completa.


## 1. Metadados

Conferir o comentário do objeto e os filtros SAP. Para views, DETAIL/HISTORY não são executados.


In [0]:
DESCRIBE EXTENDED `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`;

## 2. Volume

Reconta o volume 4128 no momento da execução.


In [0]:
SELECT COUNT(*) AS linhas_atuais, 908 AS linhas_esperadas_exploratoria FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128';

## 3. Formato do centro

Mostra eventuais variações com zeros à esquerda, sem misturar no recorte principal.


In [0]:
SELECT `cod_centro` AS centro_literal, LENGTH(TRIM(`cod_centro`)) AS tamanho, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE TRIM(`cod_centro`) RLIKE '^0*4128$' GROUP BY `cod_centro`, LENGTH(TRIM(`cod_centro`)) ORDER BY linhas DESC;

## 4. Granularidade

Confirma a chave medida na exploratória sem concatenação de valores nulos.


In [0]:
WITH d AS (SELECT `num_documento_compras`, `num_item_documento_compras` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' GROUP BY `num_documento_compras`, `num_item_documento_compras`) SELECT (SELECT COUNT(*) FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128') AS linhas, (SELECT COUNT(*) FROM d) AS chaves_distintas;

## 5. Duplicatas e versões

Distingue duplicatas idênticas de versões por hash da linha inteira.


In [0]:
WITH h AS (SELECT t.`num_documento_compras`, t.`num_item_documento_compras`, MD5(TO_JSON(STRUCT(t.`num_documento_compras`, t.`num_item_documento_compras`, t.`tp_categoria_documento_compras`, t.`tp_documento_compras`, t.`cod_material`, t.`desc_texto_breve_item`, t.`cod_centro`, t.`cod_deposito`, t.`cod_grupo_mercadorias`, t.`cod_unidade_medida_pedido`, t.`cod_unidade_medida_basica`, t.`cod_unidade_medida_preco_pedido`, t.`vl_preco_liquido`, t.`qt_unidade_preco`, t.`qt_prevista_contrato`, t.`vl_fixado_item`, t.`cod_iva`, t.`cod_domicilio_fiscal`, t.`cod_registro_info_compras`, t.`tp_categoria_item_compra`, t.`tp_classificacao_contabil`, t.`cod_eliminacao_item`, t.`cod_empresa`, t.`cod_moeda`, t.`cod_fornecedor`, t.`nm_fornecedor`, t.`cod_grupo_compradores`, t.`cod_organizacao_compras`, t.`dt_inicio_validade`, t.`dt_fim_validade`, t.`dt_documento_compras`, t.`dt_criacao_cabecalho`, t.`cod_liberacao`, t.`cod_eliminacao_cabecalho`, t.`vl_fixado_cabecalho`, t.`cod_centro_custo`, t.`num_ordem`, t.`id_interno_elemento_pep`, t.`qtd_classificacoes_contabeis`, t.`vlr_consumido`, t.`qtd_itens_pedido_consumo`, t.`ind_servico`, t.`ind_possui_cabecalho`, t.`ind_contrato_multicentro`, t.`ind_possui_consumo`, t.`dateingest`, t.`yearingest`, t.`monthingest`), MAP('ignoreNullFields','false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` t WHERE `cod_centro` = '4128'), d AS (SELECT `num_documento_compras`, `num_item_documento_compras`, COUNT(*) AS linhas, COUNT(DISTINCT hash_linha) AS versoes FROM h GROUP BY `num_documento_compras`, `num_item_documento_compras` HAVING COUNT(*) > 1) SELECT COUNT(*) AS chaves_repetidas, COALESCE(SUM(linhas-versoes),0) AS repeticoes_identicas, COUNT_IF(versoes>1) AS chaves_com_versoes FROM d;

## 6. Preenchimento de todas as colunas

Uma linha por coluna, sem expor valores pessoais. Campos vazios no universo precisam de validação no centro.


In [0]:
SELECT *, total-nulos-vazios AS preenchidos FROM (SELECT 'num_documento_compras' AS coluna, COUNT(*) AS total, COUNT_IF(`num_documento_compras` IS NULL) AS nulos, COUNT_IF(`num_documento_compras` IS NOT NULL AND LOWER(TRIM(`num_documento_compras`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_documento_compras`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_item_documento_compras' AS coluna, COUNT(*) AS total, COUNT_IF(`num_item_documento_compras` IS NULL) AS nulos, COUNT_IF(`num_item_documento_compras` IS NOT NULL AND LOWER(TRIM(`num_item_documento_compras`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_item_documento_compras`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'tp_categoria_documento_compras' AS coluna, COUNT(*) AS total, COUNT_IF(`tp_categoria_documento_compras` IS NULL) AS nulos, COUNT_IF(`tp_categoria_documento_compras` IS NOT NULL AND LOWER(TRIM(`tp_categoria_documento_compras`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`tp_categoria_documento_compras`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'tp_documento_compras' AS coluna, COUNT(*) AS total, COUNT_IF(`tp_documento_compras` IS NULL) AS nulos, COUNT_IF(`tp_documento_compras` IS NOT NULL AND LOWER(TRIM(`tp_documento_compras`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`tp_documento_compras`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_material' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_material` IS NULL) AS nulos, COUNT_IF(`cod_material` IS NOT NULL AND LOWER(TRIM(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_material`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'desc_texto_breve_item' AS coluna, COUNT(*) AS total, COUNT_IF(`desc_texto_breve_item` IS NULL) AS nulos, COUNT_IF(`desc_texto_breve_item` IS NOT NULL AND LOWER(TRIM(`desc_texto_breve_item`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`desc_texto_breve_item`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_centro' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_centro` IS NULL) AS nulos, COUNT_IF(`cod_centro` IS NOT NULL AND LOWER(TRIM(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_centro`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_deposito' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_deposito` IS NULL) AS nulos, COUNT_IF(`cod_deposito` IS NOT NULL AND LOWER(TRIM(`cod_deposito`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_deposito`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_grupo_mercadorias' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_grupo_mercadorias` IS NULL) AS nulos, COUNT_IF(`cod_grupo_mercadorias` IS NOT NULL AND LOWER(TRIM(`cod_grupo_mercadorias`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_grupo_mercadorias`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_unidade_medida_pedido' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_unidade_medida_pedido` IS NULL) AS nulos, COUNT_IF(`cod_unidade_medida_pedido` IS NOT NULL AND LOWER(TRIM(`cod_unidade_medida_pedido`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_unidade_medida_pedido`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_unidade_medida_basica' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_unidade_medida_basica` IS NULL) AS nulos, COUNT_IF(`cod_unidade_medida_basica` IS NOT NULL AND LOWER(TRIM(`cod_unidade_medida_basica`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_unidade_medida_basica`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_unidade_medida_preco_pedido' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_unidade_medida_preco_pedido` IS NULL) AS nulos, COUNT_IF(`cod_unidade_medida_preco_pedido` IS NOT NULL AND LOWER(TRIM(`cod_unidade_medida_preco_pedido`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_unidade_medida_preco_pedido`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'vl_preco_liquido' AS coluna, COUNT(*) AS total, COUNT_IF(`vl_preco_liquido` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`vl_preco_liquido`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_unidade_preco' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_unidade_preco` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`qt_unidade_preco`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_prevista_contrato' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_prevista_contrato` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`qt_prevista_contrato`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'vl_fixado_item' AS coluna, COUNT(*) AS total, COUNT_IF(`vl_fixado_item` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`vl_fixado_item`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_iva' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_iva` IS NULL) AS nulos, COUNT_IF(`cod_iva` IS NOT NULL AND LOWER(TRIM(`cod_iva`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_iva`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_domicilio_fiscal' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_domicilio_fiscal` IS NULL) AS nulos, COUNT_IF(`cod_domicilio_fiscal` IS NOT NULL AND LOWER(TRIM(`cod_domicilio_fiscal`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_domicilio_fiscal`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_registro_info_compras' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_registro_info_compras` IS NULL) AS nulos, COUNT_IF(`cod_registro_info_compras` IS NOT NULL AND LOWER(TRIM(`cod_registro_info_compras`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_registro_info_compras`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'tp_categoria_item_compra' AS coluna, COUNT(*) AS total, COUNT_IF(`tp_categoria_item_compra` IS NULL) AS nulos, COUNT_IF(`tp_categoria_item_compra` IS NOT NULL AND LOWER(TRIM(`tp_categoria_item_compra`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`tp_categoria_item_compra`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'tp_classificacao_contabil' AS coluna, COUNT(*) AS total, COUNT_IF(`tp_classificacao_contabil` IS NULL) AS nulos, COUNT_IF(`tp_classificacao_contabil` IS NOT NULL AND LOWER(TRIM(`tp_classificacao_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`tp_classificacao_contabil`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_eliminacao_item' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_eliminacao_item` IS NULL) AS nulos, COUNT_IF(`cod_eliminacao_item` IS NOT NULL AND LOWER(TRIM(`cod_eliminacao_item`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_eliminacao_item`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_empresa' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_empresa` IS NULL) AS nulos, COUNT_IF(`cod_empresa` IS NOT NULL AND LOWER(TRIM(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_empresa`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_moeda' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_moeda` IS NULL) AS nulos, COUNT_IF(`cod_moeda` IS NOT NULL AND LOWER(TRIM(`cod_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_moeda`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_fornecedor' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_fornecedor` IS NULL) AS nulos, COUNT_IF(`cod_fornecedor` IS NOT NULL AND LOWER(TRIM(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'nm_fornecedor' AS coluna, COUNT(*) AS total, COUNT_IF(`nm_fornecedor` IS NULL) AS nulos, COUNT_IF(`nm_fornecedor` IS NOT NULL AND LOWER(TRIM(`nm_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`nm_fornecedor`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_grupo_compradores' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_grupo_compradores` IS NULL) AS nulos, COUNT_IF(`cod_grupo_compradores` IS NOT NULL AND LOWER(TRIM(`cod_grupo_compradores`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_grupo_compradores`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_organizacao_compras' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_organizacao_compras` IS NULL) AS nulos, COUNT_IF(`cod_organizacao_compras` IS NOT NULL AND LOWER(TRIM(`cod_organizacao_compras`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_organizacao_compras`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'dt_inicio_validade' AS coluna, COUNT(*) AS total, COUNT_IF(`dt_inicio_validade` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'dt_fim_validade' AS coluna, COUNT(*) AS total, COUNT_IF(`dt_fim_validade` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'dt_documento_compras' AS coluna, COUNT(*) AS total, COUNT_IF(`dt_documento_compras` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'dt_criacao_cabecalho' AS coluna, COUNT(*) AS total, COUNT_IF(`dt_criacao_cabecalho` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_liberacao' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_liberacao` IS NULL) AS nulos, COUNT_IF(`cod_liberacao` IS NOT NULL AND LOWER(TRIM(`cod_liberacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_liberacao`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_eliminacao_cabecalho' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_eliminacao_cabecalho` IS NULL) AS nulos, COUNT_IF(`cod_eliminacao_cabecalho` IS NOT NULL AND LOWER(TRIM(`cod_eliminacao_cabecalho`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_eliminacao_cabecalho`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'vl_fixado_cabecalho' AS coluna, COUNT(*) AS total, COUNT_IF(`vl_fixado_cabecalho` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`vl_fixado_cabecalho`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_centro_custo' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_centro_custo` IS NULL) AS nulos, COUNT_IF(`cod_centro_custo` IS NOT NULL AND LOWER(TRIM(`cod_centro_custo`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_centro_custo`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_ordem' AS coluna, COUNT(*) AS total, COUNT_IF(`num_ordem` IS NULL) AS nulos, COUNT_IF(`num_ordem` IS NOT NULL AND LOWER(TRIM(`num_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_ordem`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'id_interno_elemento_pep' AS coluna, COUNT(*) AS total, COUNT_IF(`id_interno_elemento_pep` IS NULL) AS nulos, COUNT_IF(`id_interno_elemento_pep` IS NOT NULL AND LOWER(TRIM(`id_interno_elemento_pep`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`id_interno_elemento_pep`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qtd_classificacoes_contabeis' AS coluna, COUNT(*) AS total, COUNT_IF(`qtd_classificacoes_contabeis` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`qtd_classificacoes_contabeis`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'vlr_consumido' AS coluna, COUNT(*) AS total, COUNT_IF(`vlr_consumido` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`vlr_consumido`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qtd_itens_pedido_consumo' AS coluna, COUNT(*) AS total, COUNT_IF(`qtd_itens_pedido_consumo` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`qtd_itens_pedido_consumo`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_servico' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_servico` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_possui_cabecalho' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_possui_cabecalho` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_contrato_multicentro' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_contrato_multicentro` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_possui_consumo' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_possui_consumo` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'dateingest' AS coluna, COUNT(*) AS total, COUNT_IF(`dateingest` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'yearingest' AS coluna, COUNT(*) AS total, COUNT_IF(`yearingest` IS NULL) AS nulos, COUNT_IF(`yearingest` IS NOT NULL AND LOWER(TRIM(`yearingest`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`yearingest`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'monthingest' AS coluna, COUNT(*) AS total, COUNT_IF(`monthingest` IS NULL) AS nulos, COUNT_IF(`monthingest` IS NOT NULL AND LOWER(TRIM(`monthingest`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`monthingest`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128') a ORDER BY coluna;

## 7. Cardinalidade

Compara a cardinalidade dos identificadores e categorias não pessoais.


In [0]:
SELECT * FROM (SELECT 'num_documento_compras' AS coluna, COUNT(DISTINCT `num_documento_compras`) AS distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_item_documento_compras' AS coluna, COUNT(DISTINCT `num_item_documento_compras`) AS distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_centro' AS coluna, COUNT(DISTINCT `cod_centro`) AS distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'tp_categoria_documento_compras' AS coluna, COUNT(DISTINCT `tp_categoria_documento_compras`) AS distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'tp_documento_compras' AS coluna, COUNT(DISTINCT `tp_documento_compras`) AS distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'tp_categoria_item_compra' AS coluna, COUNT(DISTINCT `tp_categoria_item_compra`) AS distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'tp_classificacao_contabil' AS coluna, COUNT(DISTINCT `tp_classificacao_contabil`) AS distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128') a ORDER BY coluna;

## 8. Domínio

Oito valores mais comuns por categoria não pessoal.


In [0]:
WITH d AS (SELECT 'tp_categoria_documento_compras' AS coluna, CAST(`tp_categoria_documento_compras` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' GROUP BY `tp_categoria_documento_compras` UNION ALL SELECT 'tp_documento_compras' AS coluna, CAST(`tp_documento_compras` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' GROUP BY `tp_documento_compras` UNION ALL SELECT 'tp_categoria_item_compra' AS coluna, CAST(`tp_categoria_item_compra` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' GROUP BY `tp_categoria_item_compra` UNION ALL SELECT 'tp_classificacao_contabil' AS coluna, CAST(`tp_classificacao_contabil` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' GROUP BY `tp_classificacao_contabil` UNION ALL SELECT 'ind_servico' AS coluna, CAST(`ind_servico` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' GROUP BY `ind_servico` UNION ALL SELECT 'ind_possui_cabecalho' AS coluna, CAST(`ind_possui_cabecalho` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' GROUP BY `ind_possui_cabecalho`) SELECT coluna, valor, linhas FROM (SELECT d.*, ROW_NUMBER() OVER(PARTITION BY coluna ORDER BY linhas DESC, valor) AS rn FROM d) x WHERE rn<=8 ORDER BY coluna,rn;

## 9. Indicadores

Mede estados originais sem converter strings em booleanos.


In [0]:
SELECT * FROM (SELECT 'ind_servico' AS coluna, CAST(`ind_servico` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' GROUP BY `ind_servico` UNION ALL SELECT 'ind_possui_cabecalho' AS coluna, CAST(`ind_possui_cabecalho` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' GROUP BY `ind_possui_cabecalho`) a ORDER BY coluna,linhas DESC;

## 10. Perfil numérico

Quantidades e valores sem tratar códigos como medidas.


In [0]:
SELECT * FROM (SELECT 'vl_preco_liquido' AS coluna, CAST(MIN(`vl_preco_liquido`) AS DOUBLE) AS minimo, CAST(MAX(`vl_preco_liquido`) AS DOUBLE) AS maximo, CAST(AVG(`vl_preco_liquido`) AS DOUBLE) AS media, CAST(PERCENTILE_APPROX(`vl_preco_liquido`,0.5) AS DOUBLE) AS mediana, COUNT_IF(`vl_preco_liquido`<0) AS negativos, COUNT_IF(`vl_preco_liquido`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_unidade_preco' AS coluna, CAST(MIN(`qt_unidade_preco`) AS DOUBLE) AS minimo, CAST(MAX(`qt_unidade_preco`) AS DOUBLE) AS maximo, CAST(AVG(`qt_unidade_preco`) AS DOUBLE) AS media, CAST(PERCENTILE_APPROX(`qt_unidade_preco`,0.5) AS DOUBLE) AS mediana, COUNT_IF(`qt_unidade_preco`<0) AS negativos, COUNT_IF(`qt_unidade_preco`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_prevista_contrato' AS coluna, CAST(MIN(`qt_prevista_contrato`) AS DOUBLE) AS minimo, CAST(MAX(`qt_prevista_contrato`) AS DOUBLE) AS maximo, CAST(AVG(`qt_prevista_contrato`) AS DOUBLE) AS media, CAST(PERCENTILE_APPROX(`qt_prevista_contrato`,0.5) AS DOUBLE) AS mediana, COUNT_IF(`qt_prevista_contrato`<0) AS negativos, COUNT_IF(`qt_prevista_contrato`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'vl_fixado_item' AS coluna, CAST(MIN(`vl_fixado_item`) AS DOUBLE) AS minimo, CAST(MAX(`vl_fixado_item`) AS DOUBLE) AS maximo, CAST(AVG(`vl_fixado_item`) AS DOUBLE) AS media, CAST(PERCENTILE_APPROX(`vl_fixado_item`,0.5) AS DOUBLE) AS mediana, COUNT_IF(`vl_fixado_item`<0) AS negativos, COUNT_IF(`vl_fixado_item`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'vl_fixado_cabecalho' AS coluna, CAST(MIN(`vl_fixado_cabecalho`) AS DOUBLE) AS minimo, CAST(MAX(`vl_fixado_cabecalho`) AS DOUBLE) AS maximo, CAST(AVG(`vl_fixado_cabecalho`) AS DOUBLE) AS media, CAST(PERCENTILE_APPROX(`vl_fixado_cabecalho`,0.5) AS DOUBLE) AS mediana, COUNT_IF(`vl_fixado_cabecalho`<0) AS negativos, COUNT_IF(`vl_fixado_cabecalho`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qtd_classificacoes_contabeis' AS coluna, CAST(MIN(`qtd_classificacoes_contabeis`) AS DOUBLE) AS minimo, CAST(MAX(`qtd_classificacoes_contabeis`) AS DOUBLE) AS maximo, CAST(AVG(`qtd_classificacoes_contabeis`) AS DOUBLE) AS media, CAST(PERCENTILE_APPROX(`qtd_classificacoes_contabeis`,0.5) AS DOUBLE) AS mediana, COUNT_IF(`qtd_classificacoes_contabeis`<0) AS negativos, COUNT_IF(`qtd_classificacoes_contabeis`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'vlr_consumido' AS coluna, CAST(MIN(`vlr_consumido`) AS DOUBLE) AS minimo, CAST(MAX(`vlr_consumido`) AS DOUBLE) AS maximo, CAST(AVG(`vlr_consumido`) AS DOUBLE) AS media, CAST(PERCENTILE_APPROX(`vlr_consumido`,0.5) AS DOUBLE) AS mediana, COUNT_IF(`vlr_consumido`<0) AS negativos, COUNT_IF(`vlr_consumido`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qtd_itens_pedido_consumo' AS coluna, CAST(MIN(`qtd_itens_pedido_consumo`) AS DOUBLE) AS minimo, CAST(MAX(`qtd_itens_pedido_consumo`) AS DOUBLE) AS maximo, CAST(AVG(`qtd_itens_pedido_consumo`) AS DOUBLE) AS media, CAST(PERCENTILE_APPROX(`qtd_itens_pedido_consumo`,0.5) AS DOUBLE) AS mediana, COUNT_IF(`qtd_itens_pedido_consumo`<0) AS negativos, COUNT_IF(`qtd_itens_pedido_consumo`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128') a ORDER BY coluna;

## 11. Totais para conciliação

Totais por campo; verificar moeda, granularidade e DOUBLE.


In [0]:
SELECT * FROM (SELECT 'vl_preco_liquido' AS coluna, CAST(SUM(`vl_preco_liquido`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_unidade_preco' AS coluna, CAST(SUM(`qt_unidade_preco`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_prevista_contrato' AS coluna, CAST(SUM(`qt_prevista_contrato`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'vl_fixado_item' AS coluna, CAST(SUM(`vl_fixado_item`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'vl_fixado_cabecalho' AS coluna, CAST(SUM(`vl_fixado_cabecalho`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qtd_classificacoes_contabeis' AS coluna, CAST(SUM(`qtd_classificacoes_contabeis`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'vlr_consumido' AS coluna, CAST(SUM(`vlr_consumido`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qtd_itens_pedido_consumo' AS coluna, CAST(SUM(`qtd_itens_pedido_consumo`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128') a ORDER BY coluna;

## 12. Datas

Min/max de strings são literais, não ordem cronológica.


In [0]:
SELECT * FROM (SELECT 'dt_inicio_validade' AS coluna, CAST(MIN(`dt_inicio_validade`) AS STRING) AS menor_literal, CAST(MAX(`dt_inicio_validade`) AS STRING) AS maior_literal, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'dt_fim_validade' AS coluna, CAST(MIN(`dt_fim_validade`) AS STRING) AS menor_literal, CAST(MAX(`dt_fim_validade`) AS STRING) AS maior_literal, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'dt_documento_compras' AS coluna, CAST(MIN(`dt_documento_compras`) AS STRING) AS menor_literal, CAST(MAX(`dt_documento_compras`) AS STRING) AS maior_literal, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'dt_criacao_cabecalho' AS coluna, CAST(MIN(`dt_criacao_cabecalho`) AS STRING) AS menor_literal, CAST(MAX(`dt_criacao_cabecalho`) AS STRING) AS maior_literal, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'dateingest' AS coluna, CAST(MIN(`dateingest`) AS STRING) AS menor_literal, CAST(MAX(`dateingest`) AS STRING) AS maior_literal, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128') a ORDER BY coluna;

## 13. Zeros à esquerda

Mede códigos numéricos com prefixo zero sem alterar a fonte.


In [0]:
SELECT * FROM (SELECT 'num_documento_compras' AS coluna, COUNT_IF(TRIM(`num_documento_compras`) RLIKE '^0+[0-9]+$') AS com_zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_item_documento_compras' AS coluna, COUNT_IF(TRIM(`num_item_documento_compras`) RLIKE '^0+[0-9]+$') AS com_zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_material' AS coluna, COUNT_IF(TRIM(`cod_material`) RLIKE '^0+[0-9]+$') AS com_zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_centro' AS coluna, COUNT_IF(TRIM(`cod_centro`) RLIKE '^0+[0-9]+$') AS com_zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_deposito' AS coluna, COUNT_IF(TRIM(`cod_deposito`) RLIKE '^0+[0-9]+$') AS com_zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_grupo_mercadorias' AS coluna, COUNT_IF(TRIM(`cod_grupo_mercadorias`) RLIKE '^0+[0-9]+$') AS com_zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128') a ORDER BY coluna;

## 14. Checksum

Duplicatas idênticas sem divulgar hash ou dados.


In [0]:
WITH h AS (SELECT MD5(TO_JSON(STRUCT(t.`num_documento_compras`, t.`num_item_documento_compras`, t.`tp_categoria_documento_compras`, t.`tp_documento_compras`, t.`cod_material`, t.`desc_texto_breve_item`, t.`cod_centro`, t.`cod_deposito`, t.`cod_grupo_mercadorias`, t.`cod_unidade_medida_pedido`, t.`cod_unidade_medida_basica`, t.`cod_unidade_medida_preco_pedido`, t.`vl_preco_liquido`, t.`qt_unidade_preco`, t.`qt_prevista_contrato`, t.`vl_fixado_item`, t.`cod_iva`, t.`cod_domicilio_fiscal`, t.`cod_registro_info_compras`, t.`tp_categoria_item_compra`, t.`tp_classificacao_contabil`, t.`cod_eliminacao_item`, t.`cod_empresa`, t.`cod_moeda`, t.`cod_fornecedor`, t.`nm_fornecedor`, t.`cod_grupo_compradores`, t.`cod_organizacao_compras`, t.`dt_inicio_validade`, t.`dt_fim_validade`, t.`dt_documento_compras`, t.`dt_criacao_cabecalho`, t.`cod_liberacao`, t.`cod_eliminacao_cabecalho`, t.`vl_fixado_cabecalho`, t.`cod_centro_custo`, t.`num_ordem`, t.`id_interno_elemento_pep`, t.`qtd_classificacoes_contabeis`, t.`vlr_consumido`, t.`qtd_itens_pedido_consumo`, t.`ind_servico`, t.`ind_possui_cabecalho`, t.`ind_contrato_multicentro`, t.`ind_possui_consumo`, t.`dateingest`, t.`yearingest`, t.`monthingest`), MAP('ignoreNullFields','false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` t WHERE `cod_centro` = '4128'), d AS (SELECT hash_linha,COUNT(*) AS linhas FROM h GROUP BY hash_linha) SELECT COALESCE(SUM(linhas),0) AS linhas, COUNT(*) AS distintos, COALESCE(SUM(linhas-1),0) AS duplicatas_identicas FROM d;

## 15. Amostra

Sem dados pessoais literais.


In [0]:
SELECT `num_documento_compras`, `num_item_documento_compras`, `tp_categoria_documento_compras`, `tp_documento_compras`, `cod_material`, `desc_texto_breve_item`, `cod_centro`, `cod_deposito`, `cod_grupo_mercadorias`, `cod_unidade_medida_pedido`, `cod_unidade_medida_basica`, `cod_unidade_medida_preco_pedido`, `vl_preco_liquido`, `qt_unidade_preco`, `qt_prevista_contrato`, `vl_fixado_item`, `cod_iva`, `cod_domicilio_fiscal`, `cod_registro_info_compras`, `tp_categoria_item_compra`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' LIMIT 20;

## 16. Distribuição

Dimensões de negócio sem pessoas.


In [0]:
SELECT `cod_centro`, `tp_categoria_documento_compras`, `tp_documento_compras`,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' GROUP BY `cod_centro`, `tp_categoria_documento_compras`, `tp_documento_compras` ORDER BY linhas DESC LIMIT 100;

## 17. Freshness

Snapshot versus data de negócio.


In [0]:
SELECT COUNT(*) AS linhas,MIN(`dateingest`) AS `min_dateingest`, MAX(`dateingest`) AS `max_dateingest`, MIN(`yearingest`) AS `min_yearingest`, MAX(`yearingest`) AS `max_yearingest`, MIN(`monthingest`) AS `min_monthingest`, MAX(`monthingest`) AS `max_monthingest` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128';

## 18. Regras específicas

Testes derivados do DESCRIBE, não conclusões.


In [0]:
SELECT COUNT(*) AS linhas,COUNT_IF(`dt_fim_validade`<`dt_inicio_validade`) AS vigencia_invertida,COUNT_IF(`ind_possui_cabecalho`=FALSE) AS sem_cabecalho FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128';

## 19. Amostra ampliada

Até 500 linhas sem dados pessoais literais.


In [0]:
SELECT `num_documento_compras`, `num_item_documento_compras`, `tp_categoria_documento_compras`, `tp_documento_compras`, `cod_material`, `desc_texto_breve_item`, `cod_centro`, `cod_deposito`, `cod_grupo_mercadorias`, `cod_unidade_medida_pedido`, `cod_unidade_medida_basica`, `cod_unidade_medida_preco_pedido`, `vl_preco_liquido`, `qt_unidade_preco`, `qt_prevista_contrato`, `vl_fixado_item`, `cod_iva`, `cod_domicilio_fiscal`, `cod_registro_info_compras`, `tp_categoria_item_compra`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' LIMIT 500;

## 20. Resumo

Volume atual e chave observada no universo.


In [0]:
WITH d AS (SELECT `num_documento_compras`, `num_item_documento_compras`,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' GROUP BY `num_documento_compras`, `num_item_documento_compras`) SELECT 'ME3N' AS transacao,'4128' AS centro,908 AS volume_esperado,(SELECT COUNT(*) FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128') AS volume_atual,(SELECT COUNT(*) FROM d) AS chaves_distintas,(SELECT COUNT(*) FROM d WHERE linhas>1) AS chaves_repetidas,'num_documento_compras + num_item_documento_compras' AS chave_observada;

## 21. EXTRAÇÃO COMPLETA da base 4128

Sem LIMIT, sem dados pessoais literais. Comparar com SAP apenas após confirmar filtros da tela.


In [0]:
SELECT CASE WHEN TRIM(CAST(`num_documento_compras` AS STRING)) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(CAST(`num_documento_compras` AS STRING)),'^0+(?=[0-9])','') ELSE TRIM(CAST(`num_documento_compras` AS STRING)) END AS `num_documento_compras_normalizado`, CASE WHEN TRIM(CAST(`num_item_documento_compras` AS STRING)) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(CAST(`num_item_documento_compras` AS STRING)),'^0+(?=[0-9])','') ELSE TRIM(CAST(`num_item_documento_compras` AS STRING)) END AS `num_item_documento_compras_normalizado`, `num_documento_compras`, `num_item_documento_compras`, `tp_categoria_documento_compras`, `tp_documento_compras`, `cod_material`, `desc_texto_breve_item`, `cod_centro`, `cod_deposito`, `cod_grupo_mercadorias`, `cod_unidade_medida_pedido`, `cod_unidade_medida_basica`, `cod_unidade_medida_preco_pedido`, `vl_preco_liquido`, `qt_unidade_preco`, `qt_prevista_contrato`, `vl_fixado_item`, `cod_iva`, `cod_domicilio_fiscal`, `cod_registro_info_compras`, `tp_categoria_item_compra`, `tp_classificacao_contabil`, `cod_eliminacao_item`, `cod_empresa`, `cod_moeda`, `cod_fornecedor`, `cod_grupo_compradores`, `cod_organizacao_compras`, `dt_inicio_validade`, `dt_fim_validade`, `dt_documento_compras`, `dt_criacao_cabecalho`, `cod_liberacao`, `cod_eliminacao_cabecalho`, `vl_fixado_cabecalho`, `cod_centro_custo`, `num_ordem`, `id_interno_elemento_pep`, `qtd_classificacoes_contabeis`, `vlr_consumido`, `qtd_itens_pedido_consumo`, `ind_servico`, `ind_possui_cabecalho`, `ind_contrato_multicentro`, `ind_possui_consumo`, `dateingest`, `yearingest`, `monthingest`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE `cod_centro` = '4128' ORDER BY `num_documento_compras`, `num_item_documento_compras`;

## Checklist SAP

- [ ] Confirmar centro 4128, janela temporal e filtros da tela.
- [ ] Executar no mesmo dia da extração SAP.
- [ ] Conferir chave, duplicatas, zeros à esquerda, datas, moeda e DOUBLE.
- [ ] Validar campos 100% sem valor com o dono do dado.
- [ ] Não exportar dados pessoais literais.
